# AI 5~9 분석 워크북 (Refactor)

요청하신 대로 **주피터 노트북 기반**으로 1~5 분석을 실행/점검하고, 핵심 지표를 시각화합니다.

- 분석 1: AI가입일 = 전자금융가입일 정합 + 이벤트 전후 추이
- 분석 2: AI가입 전/후 이체 비교 (노출기간 보정 포함)
- 분석 3: 재사용률(가입시점 차이 고려)
- 분석 4: 재사용자 특징 가설 검증
- 분석 5: 미사용/미재사용 원인 가설 검증

> 컬럼명이 최종 확정되면 `common.py`의 매핑 후보만 추가하면 동일 로직으로 재실행 가능합니다.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

root = Path.cwd()
refactor_dir = root / "analysis" / "refactor"
if not refactor_dir.exists() and root.name == "refactor":
    refactor_dir = root
if str(refactor_dir) not in sys.path:
    sys.path.insert(0, str(refactor_dir))

import common
import a01_signup_alignment as a01
import a02_transfer_prepost as a02
import a03_reuse_rate as a03
import a04_reuser_characteristics as a04
import a05_nonreuse_causes as a05

print(f"refactor_dir = {refactor_dir}")

## 0) 입력 경로 설정

아래 파일 경로를 실제 데이터 파일로 바꾼 뒤 실행하세요.

In [ ]:
PROFILE_FILE = "data/customer_profile.csv"
CHAT_FILE = "data/ai_chat_daily_by_user.csv"
AI_TRANSFER_FILE = "data/ai_transfer_daily_by_user.csv"
EVENT_FILE = "data/event_calendar.csv"  # 없으면 None으로 변경

OPEN_DATE = "2026-03-23"
OUTPUT_DIR = Path("output_5to9_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REUSE_MIN_DAYS = 2
REUSE_MIN_REQ = 3
NONREUSE_MAX_REQ = 2
SENIOR_AGE = 60

print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
profile_raw = common.load_csv(PROFILE_FILE)
chat_raw = common.load_csv(CHAT_FILE)
ai_transfer_raw = common.load_csv(AI_TRANSFER_FILE)
event_raw = common.load_csv(EVENT_FILE) if EVENT_FILE else None

profile = common.normalize_profile(profile_raw)
chat = common.normalize_chat(chat_raw)
ai_transfer = common.normalize_ai_transfer(ai_transfer_raw)
events = common.normalize_event(event_raw)

print("profile:", profile.shape)
print("chat:", chat.shape)
print("ai_transfer:", ai_transfer.shape)
print("events:", events.shape if isinstance(events, pd.DataFrame) else None)

display(profile.head(2))
display(chat.head(2))

## 1) AI가입일 = 전자금융가입일 정합 + 이벤트 전후 검증

In [ ]:
alignment = a01.analyze_signup_alignment(profile)
daily = a01.build_daily_from_profile(profile)
event_table = a01.analyze_event_windows(daily, events) if isinstance(events, pd.DataFrame) and not events.empty else pd.DataFrame()

# 저장
common.save_csv(alignment, OUTPUT_DIR, "a01_signup_alignment_notebook.csv")
common.save_csv(daily, OUTPUT_DIR, "a01_daily_notebook.csv")
if not event_table.empty:
    common.save_csv(event_table, OUTPUT_DIR, "a01_event_window_notebook.csv")

display(Markdown("### 1-A. AI가입일 vs 전자금융가입일"))
display(alignment)

display(Markdown("### 1-B. 이벤트 전후 비교"))
display(event_table)

print("해석가이드:", a01.suggest_comment(alignment))

if not daily.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.lineplot(data=daily, x="date", y="new_signups", marker="o", label="신규가입", ax=axes[0])
    sns.lineplot(data=daily, x="date", y="ai_signups", marker="o", label="AI가입", ax=axes[0])
    axes[0].set_title("일자별 신규가입 / AI가입")
    axes[0].tick_params(axis="x", rotation=30)

    sns.lineplot(data=daily, x="date", y="same_day_conv_rate", marker="o", color="#dd8452", ax=axes[1])
    axes[1].set_title("동일일 전환율(정확매칭)")
    axes[1].set_ylim(bottom=0)
    axes[1].tick_params(axis="x", rotation=30)

    if isinstance(events, pd.DataFrame) and not events.empty and "event_date" in events.columns:
        for _, r in events.dropna(subset=["event_date"]).iterrows():
            for ax in axes:
                ax.axvline(r["event_date"], color="gray", linestyle="--", alpha=0.35)

    plt.tight_layout()
    plt.show()

## 2) AI가입 전/후 이체 유형별 비교 (노출기간 고려)

In [ ]:
asof_date = chat["chat_date"].max() if "chat_date" in chat.columns and not chat.empty else pd.to_datetime(OPEN_DATE)

metrics2 = a02.build_transfer_window_metrics(profile)
exposure2 = a02.age_exposure_table(profile, asof_date)
funnel2 = a02.build_funnel(profile, chat, ai_transfer)
interp2 = a02.interpretation(metrics2, exposure2, funnel2)

common.save_csv(metrics2, OUTPUT_DIR, "a02_transfer_metrics_notebook.csv")
common.save_csv(exposure2, OUTPUT_DIR, "a02_exposure_notebook.csv")
common.save_csv(funnel2, OUTPUT_DIR, "a02_funnel_notebook.csv")

print(interp2)
display(metrics2)
display(exposure2)
display(funnel2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=funnel2, x="stage", y="ratio_vs_signup", color="#4c72b0", ax=axes[0])
axes[0].set_title("AI가입자 대비 퍼널 전환률")
axes[0].tick_params(axis="x", rotation=20)

plot_m = metrics2.copy()
plot_m = plot_m[plot_m["metric"].str.contains("평균", na=False)]
sns.barplot(data=plot_m, y="metric", x="value", palette="viridis", ax=axes[1])
axes[1].set_title("전/후 이체 평균 지표")

plt.tight_layout()
plt.show()

In [ ]:
display(Markdown("## 3) AI가입자 재사용률 (가입시점 차이 반영)"))

ai_base = profile[["customer_id", "ai_signup_date"]].dropna(subset=["customer_id", "ai_signup_date"]).drop_duplicates("customer_id")
chat3 = chat.dropna(subset=["customer_id", "chat_date"]).merge(ai_base, on="customer_id", how="inner")
chat3["day_offset"] = (chat3["chat_date"] - chat3["ai_signup_date"]).dt.days
chat3 = chat3[chat3["day_offset"] >= 0].copy()

user_agg = chat3.groupby("customer_id", as_index=False).agg(
    total_requests=("request_count", "sum"),
    use_days=("chat_date", "nunique"),
)
user3 = ai_base.merge(user_agg, on="customer_id", how="left")
user3[["total_requests", "use_days"]] = user3[["total_requests", "use_days"]].fillna(0)
user3["is_reuser"] = user3["use_days"] >= REUSE_MIN_DAYS

base_n = user3["customer_id"].nunique()
summary3 = pd.DataFrame([
    {"지표": "AI가입자수", "값": base_n},
    {"지표": "가입 후 1회 이상 사용자", "값": int((user3["total_requests"] > 0).sum())},
    {"지표": f"{REUSE_MIN_DAYS}일 이상 사용자(재사용)", "값": int(user3["is_reuser"].sum())},
])
summary3["비율"] = summary3["값"].apply(lambda x: common.safe_div(x, base_n))
common.save_csv(summary3, OUTPUT_DIR, "a03_reuse_summary_notebook.csv")
display(summary3)

curve_rows = []
for d in range(1, 31):
    tmp = chat3[chat3["day_offset"] <= d]
    by_user = tmp.groupby("customer_id", as_index=False).agg(use_days=("chat_date", "nunique"))
    reuse_users = int((by_user["use_days"] >= REUSE_MIN_DAYS).sum())
    curve_rows.append({"day": d, "cum_reuse_rate": common.safe_div(reuse_users, base_n)})
curve3 = pd.DataFrame(curve_rows)
common.save_csv(curve3, OUTPUT_DIR, "a03_reuse_curve_notebook.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=summary3.iloc[1:], x="지표", y="비율", palette="Set2", ax=axes[0])
axes[0].set_title("재사용 핵심 비율")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_ylim(0, 1)

sns.lineplot(data=curve3, x="day", y="cum_reuse_rate", marker="o", color="#55a868", ax=axes[1])
axes[1].set_title("가입 후 누적 재사용률 (1~30일)")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4) 재사용자 특징 가설 검증 (STT/연령/메뉴/이체)

In [ ]:
saved4 = a04.run(
    profile_file=PROFILE_FILE,
    chat_file=CHAT_FILE,
    out_dir=OUTPUT_DIR,
    reuse_min_req=REUSE_MIN_REQ,
    senior_age=SENIOR_AGE,
)

result4 = pd.read_csv(saved4["a04_reuser_hypothesis_results"], encoding="utf-8-sig")
verdict4 = pd.read_csv(saved4["a04_reuser_hypothesis_verdict"], encoding="utf-8-sig")

display(result4)
display(verdict4)

plot4 = result4.copy().dropna(subset=["diff"])
if not plot4.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=plot4.sort_values("diff", ascending=False), x="diff", y="feature", palette="coolwarm")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("재사용군 - 미재사용군 평균 차이")
    plt.tight_layout()
    plt.show()

In [ ]:
display(Markdown("## 5) 미사용/미재사용 원인 가설 검증"))

non_label, reuse_label = a05.group_labels(NONREUSE_MAX_REQ)
reuse5 = a05.build_reuse_group(profile, chat, NONREUSE_MAX_REQ)
unans5 = a05.build_unanswered_features(chat)

base5 = profile.merge(reuse5, on="customer_id", how="inner")
base5 = base5.merge(unans5, on="customer_id", how="left")

if "is_churned" in base5.columns:
    base5["is_churned"] = base5["is_churned"].astype(str).str.strip().str.upper().isin(["Y", "TRUE", "1"])
elif "withdrawn_yn" in base5.columns:
    base5["is_churned"] = base5["withdrawn_yn"].astype(str).str.strip().str.upper().eq("Y")

t_n1 = a05.test_n1(base5, non_label, reuse_label)
t_n2 = a05.test_n2(base5, non_label, reuse_label)
t_n3 = a05.test_n3(base5, non_label, reuse_label)
result5 = pd.concat([t_n1, t_n2, t_n3], ignore_index=True)

common.save_csv(result5, OUTPUT_DIR, "a05_nonreuse_hypothesis_notebook.csv")
display(result5)

plot5 = result5.dropna(subset=["diff_nonreuse_minus_reuse"]).copy()
if not plot5.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=plot5.sort_values("diff_nonreuse_minus_reuse", ascending=False),
        x="diff_nonreuse_minus_reuse",
        y="metric",
        palette="mako",
    )
    plt.axvline(0, color="black", linewidth=1)
    plt.title("미재사용군 - 재사용군 차이")
    plt.tight_layout()
    plt.show()

## 6) 자동 해석 코멘트(초안)

아래 코드는 결과값 기반으로 보고서 문구 초안을 자동 생성합니다. (최종 문구는 수동 보정 권장)

In [ ]:
comments = []

# 1) 동일일 가입 비중
if not alignment.empty and "구간" in alignment.columns:
    same_ratio_s = alignment.loc[alignment["구간"] == "동일일(0일)", "비율"]
    same_ratio = float(same_ratio_s.iloc[0]) if len(same_ratio_s) else np.nan
    if pd.notna(same_ratio):
        if same_ratio >= 0.2:
            comments.append(f"동일일 가입 비중 {same_ratio:.1%}: 신규유입 동시진입 신호가 유의미합니다.")
        else:
            comments.append(f"동일일 가입 비중 {same_ratio:.1%}: 기존사용자의 탐색적 수용 패턴이 상대적으로 우세합니다.")

# 3) 재사용률
reuse_row = summary3.loc[summary3["지표"].str.contains("재사용", na=False), "비율"]
reuse_rate = float(reuse_row.iloc[0]) if len(reuse_row) else np.nan
if pd.notna(reuse_rate):
    if reuse_rate >= 0.25:
        comments.append(f"재사용률 {reuse_rate:.1%}: 이체/메뉴 중심 반복사용 신호가 확인됩니다.")
    elif reuse_rate >= 0.12:
        comments.append(f"재사용률 {reuse_rate:.1%}: 초기 수용은 있으나 재방문 유도 개선이 필요합니다.")
    else:
        comments.append(f"재사용률 {reuse_rate:.1%}: 첫 성공경험 설계 및 핵심 니즈(예: 대출, FAQ 응답강화) 보강이 우선입니다.")

# 5) 답변불가/여신
if not result5.empty:
    key_n1 = result5[result5["metric"].str.contains("첫응답", na=False)]
    if not key_n1.empty:
        d = key_n1["diff_nonreuse_minus_reuse"].iloc[0]
        if pd.notna(d) and d > 0.05:
            comments.append("미재사용군에서 초기 답변불가 경험이 상대적으로 높아, 응답품질 개선이 재사용 전환에 중요합니다.")

    key_n2 = result5[result5["hypothesis"] == "N2"]
    if not key_n2.empty:
        d = key_n2["diff_nonreuse_minus_reuse"].iloc[0]
        if pd.notna(d) and d > 0:
            comments.append("미재사용군 내 여신고객 비중이 높아, 대출 관련 시나리오 확장이 필요합니다.")

for i, cmt in enumerate(comments, start=1):
    print(f"{i}. {cmt}")